# Visualisations pour la prédiction TMRT


Ce notebook regroupe les figures utiles pour présenter la partie prédiction du semestre 2. Il ne lance pas de script externe : chaque figure est construite ici, à partir des tableaux exportés par `Prediction.ipynb`.

L'objectif est de garder des graphiques simples et lisibles, dans le même esprit que les figures d'analyse exploratoire : fond blanc, titres courts, couleurs sobres et sorties en `dpi=150`.


In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed" / "prediction"
REPORT_DIR = DATA_DIR / "tmrt_pred_report"
EVAL_DIR = DATA_DIR / "eval"
BIAS_DIR = DATA_DIR / "biais"
NO_IID_DIR = DATA_DIR / "no_iid"

FIG_ROOT = ROOT / "prediction" / "figures"
FIG_PRED_DATA = FIG_ROOT / "prediction_data"
FIG_EVAL = FIG_ROOT / "eval"
FIG_BIAS = FIG_ROOT / "biais"
FIG_MODEL = FIG_ROOT / "model_analysis"
FIG_NO_IID = FIG_ROOT / "no_iid"

for folder in [FIG_PRED_DATA, FIG_EVAL, FIG_BIAS, FIG_MODEL, FIG_NO_IID]:
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
RED = "#d62728"
GREY = "#666666"
TRACK_ORDER = ["antigone", "boulevards", "ecusson"]
MSLOT_ORDER = ["M1", "M2", "M3", "M4"]
SPLIT_ORDER = ["random_naive", "spatial_section_group", "passage_group", "leave_one_track_out", "date_group"]

GENERATED_FIGURES = []


def rel(path):
    path = Path(path)
    try:
        return path.relative_to(ROOT)
    except ValueError:
        return path


def save_figure(fig, path, extra_paths=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    GENERATED_FIGURES.append(path)

    for extra_path in extra_paths or []:
        extra_path = Path(extra_path)
        extra_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(extra_path, dpi=150, bbox_inches="tight")
        GENERATED_FIGURES.append(extra_path)

    if "agg" in plt.get_backend().lower():
        plt.close(fig)
    else:
        plt.show()
    print("Figure enregistrée :", rel(path))
    if extra_paths:
        for extra_path in extra_paths:
            print("Copie enregistrée :", rel(extra_path))


def ordered_existing(values, preferred):
    values = [v for v in preferred if v in set(values)]
    return values

def boxplot_with_labels(ax, values, labels, **kwargs):
    try:
        return ax.boxplot(values, tick_labels=labels, **kwargs)
    except TypeError:
        return ax.boxplot(values, labels=labels, **kwargs)


def numeric_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def read_csv_if_exists(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print("Fichier absent :", rel(path))
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


## Chargement des données

On repart des exports disponibles dans `data/processed/prediction/`. Le tableau principal contient les observations TMRT et les prédictions associées. Les autres fichiers servent à analyser les performances, les biais et la structure des groupes de passage.


In [ ]:
df_pred = read_csv_if_exists(DATA_DIR / "tmrt_predictions_grid_or_points.csv")

# Pour l'analyse des erreurs, on utilise en priorité les lignes de test.
test_rows = read_csv_if_exists(
    REPORT_DIR / "tmrt_pred_testrows_MF_plus_AE_raw64.csv",
    sep=";",
    decimal=",",
)

if test_rows.empty:
    test_rows = read_csv_if_exists(REPORT_DIR / "tmrt_pred_MF_plus_AE_raw64.csv")

model_comparison = read_csv_if_exists(EVAL_DIR / "model_comparison.csv")
metrics_all = read_csv_if_exists(EVAL_DIR / "evaluation_metrics_all_splits.csv")
metrics_by_group = read_csv_if_exists(EVAL_DIR / "evaluation_metrics_by_group.csv")
bias_analysis = read_csv_if_exists(BIAS_DIR / "bias_analysis.csv")
no_iid_diag = read_csv_if_exists(NO_IID_DIR / "diagnostics.csv")
group_errors = read_csv_if_exists(REPORT_DIR / "tmrt_pred_error_by_groups.csv")

for frame in [df_pred, test_rows, group_errors, metrics_all, metrics_by_group, bias_analysis, model_comparison]:
    if not frame.empty:
        numeric_columns(
            frame,
            [
                "tmrt", "tmrt_pred", "err", "mae", "rmse", "r2", "bias",
                "mae_mean", "rmse_mean", "r2_mean", "bias_mean",
                "bias_mean_pred_minus_obs", "median_error_pred_minus_obs",
                "n", "n_rows", "n_test", "n_test_total",
                "bias_ci95_low_group_bootstrap", "bias_ci95_high_group_bootstrap",
                "lon_map", "lat_map", "lon_ontrack", "lat_ontrack",
            ],
        )
        if "err" not in frame.columns and {"tmrt", "tmrt_pred"}.issubset(frame.columns):
            frame["err"] = frame["tmrt_pred"] - frame["tmrt"]

analysis_df = df_pred if not df_pred.empty else test_rows

resume = []
for name, frame in [
    ("prédictions", df_pred),
    ("lignes avec erreur", test_rows),
    ("comparaison des splits", model_comparison),
    ("métriques par groupe", metrics_by_group),
    ("biais", bias_analysis),
    ("diagnostic non-iid", no_iid_diag),
]:
    resume.append({"table": name, "lignes": len(frame), "colonnes": len(frame.columns) if not frame.empty else 0})

display(pd.DataFrame(resume))


## 1. Données utilisées pour la prédiction

Avant d'analyser les scores, on vérifie ce que le modèle voit : distribution de la TMRT, différences entre parcours, créneaux horaires et cohérence globale entre observation et prédiction.


In [ ]:
if analysis_df.empty or not {"tmrt", "tmrt_pred"}.issubset(analysis_df.columns):
    print("Données insuffisantes pour produire la vue d'ensemble.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    obs = analysis_df["tmrt"].dropna()
    pred = analysis_df["tmrt_pred"].dropna()

    axes[0, 0].hist(obs, bins=50, alpha=0.65, label="Observation", color=BLUE)
    axes[0, 0].hist(pred, bins=50, alpha=0.45, label="Prédiction", color=ORANGE)
    axes[0, 0].set_title("Distribution de la TMRT")
    axes[0, 0].set_xlabel("TMRT (°C)")
    axes[0, 0].set_ylabel("Nombre de lignes")
    axes[0, 0].legend()

    if "track_id" in analysis_df.columns:
        tracks = ordered_existing(analysis_df["track_id"].dropna().unique(), TRACK_ORDER)
        values = [analysis_df.loc[analysis_df["track_id"].eq(track), "tmrt"].dropna() for track in tracks]
        boxplot_with_labels(axes[0, 1], values, tracks, showfliers=False)
        axes[0, 1].set_title("TMRT observée par parcours")
        axes[0, 1].set_xlabel("Parcours")
        axes[0, 1].set_ylabel("TMRT (°C)")
    else:
        axes[0, 1].axis("off")

    if "M_slot" in analysis_df.columns:
        slots = ordered_existing(analysis_df["M_slot"].dropna().unique(), MSLOT_ORDER)
        values = [analysis_df.loc[analysis_df["M_slot"].eq(slot), "tmrt"].dropna() for slot in slots]
        boxplot_with_labels(axes[1, 0], values, slots, showfliers=False)
        axes[1, 0].set_title("TMRT observée par créneau")
        axes[1, 0].set_xlabel("Créneau")
        axes[1, 0].set_ylabel("TMRT (°C)")
    else:
        axes[1, 0].axis("off")

    sample = analysis_df.dropna(subset=["tmrt", "tmrt_pred"]).sample(
        n=min(30000, len(analysis_df.dropna(subset=["tmrt", "tmrt_pred"]))),
        random_state=42,
    )
    axes[1, 1].scatter(sample["tmrt"], sample["tmrt_pred"], s=3, alpha=0.25, color=BLUE)
    low = min(sample["tmrt"].min(), sample["tmrt_pred"].min())
    high = max(sample["tmrt"].max(), sample["tmrt_pred"].max())
    axes[1, 1].plot([low, high], [low, high], color="black", linewidth=1)
    axes[1, 1].set_title("Prédiction vs observation")
    axes[1, 1].set_xlabel("TMRT observée (°C)")
    axes[1, 1].set_ylabel("TMRT prédite (°C)")

    save_figure(fig, FIG_PRED_DATA / "prediction_data_overview.png")


In [ ]:
if analysis_df.empty or not {"track_id", "M_slot", "tmrt"}.issubset(analysis_df.columns):
    print("Colonnes track_id, M_slot ou tmrt absentes.")
else:
    tracks = ordered_existing(analysis_df["track_id"].dropna().unique(), TRACK_ORDER)
    slots = ordered_existing(analysis_df["M_slot"].dropna().unique(), MSLOT_ORDER)
    fig, axes = plt.subplots(1, len(tracks), figsize=(4.5 * len(tracks), 4), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, track in zip(axes, tracks):
        part = analysis_df[analysis_df["track_id"].eq(track)]
        values = [part.loc[part["M_slot"].eq(slot), "tmrt"].dropna() for slot in slots]
        boxplot_with_labels(ax, values, slots, showfliers=False)
        ax.set_title(track)
        ax.set_xlabel("Créneau")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("TMRT observée (°C)")
    fig.suptitle("TMRT observée par parcours et par créneau", y=1.03)
    save_figure(fig, FIG_PRED_DATA / "tmrt_by_track_mslot.png")


## 2. Évaluation du modèle

Ces figures servent à comparer les performances selon la manière de séparer les données. Le split aléatoire reste utile comme repère, mais les splits par passage, section, date ou parcours donnent une lecture plus prudente de la généralisation.


In [ ]:
plot_metrics = model_comparison.copy()
if plot_metrics.empty and not metrics_all.empty:
    plot_metrics = (
        metrics_all.groupby(["feature_set", "model", "split"], as_index=False)
        .agg(
            rmse_mean=("rmse", "mean"),
            mae_mean=("mae", "mean"),
            r2_mean=("r2", "mean"),
        )
    )

if plot_metrics.empty:
    print("Aucune table de comparaison des modèles disponible.")
else:
    plot_metrics = plot_metrics[plot_metrics["feature_set"].eq("feature_set_generalizable")].copy()
    split_order = [s for s in SPLIT_ORDER if s in set(plot_metrics["split"])]
    models = [m for m in ["Ridge", "HistGradientBoosting"] if m in set(plot_metrics["model"])]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    specs = [
        ("rmse_mean", "RMSE (°C)"),
        ("mae_mean", "MAE (°C)"),
        ("r2_mean", "R²"),
    ]
    x = np.arange(len(split_order))
    width = 0.36 if len(models) > 1 else 0.55
    colors = [BLUE, ORANGE]

    for ax, (metric, label) in zip(axes, specs):
        for i, model in enumerate(models):
            sub = plot_metrics[plot_metrics["model"].eq(model)].set_index("split")
            vals = [sub.loc[s, metric] if s in sub.index else np.nan for s in split_order]
            offset = (i - (len(models) - 1) / 2) * width
            ax.bar(x + offset, vals, width=width, label=model, color=colors[i % len(colors)])
        ax.set_xticks(x)
        ax.set_xticklabels(split_order, rotation=25, ha="right")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(axis="y", alpha=0.25)

    axes[0].legend()
    fig.suptitle("Comparaison des performances selon le split", y=1.03)
    save_figure(fig, FIG_EVAL / "metrics_comparison_splits.png")


In [ ]:
if plot_metrics.empty:
    print("Comparaison des splits non disponible.")
else:
    hgb = plot_metrics[plot_metrics["model"].eq("HistGradientBoosting")].set_index("split")
    if "random_naive" not in hgb.index:
        print("Le split random_naive est absent : impossible de calculer l'écart de généralisation.")
    else:
        split_order = [s for s in SPLIT_ORDER if s in hgb.index]
        baseline = hgb.loc["random_naive", "rmse_mean"]
        gaps = [hgb.loc[s, "rmse_mean"] - baseline for s in split_order]

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(split_order, gaps, color=BLUE)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylabel("Écart de RMSE vs split aléatoire (°C)")
        ax.set_title("Perte de performance avec des splits plus réalistes")
        ax.tick_params(axis="x", rotation=25)
        ax.grid(axis="y", alpha=0.25)

        save_figure(fig, FIG_EVAL / "generalization_gap_by_split.png")


In [ ]:
split_predictions_path = EVAL_DIR / "predictions_by_split.csv"

if split_predictions_path.exists():
    pred_split = pd.read_csv(split_predictions_path)
    numeric_columns(pred_split, ["tmrt", "tmrt_pred"])
    pred_split = pred_split.dropna(subset=["tmrt", "tmrt_pred", "split"])
else:
    pred_split = test_rows if not test_rows.empty else analysis_df
    pred_split = pred_split.dropna(subset=["tmrt", "tmrt_pred"]).copy()
    pred_split["split"] = "prédictions disponibles"
    print("Pas de fichier predictions_by_split.csv : la figure utilise les prédictions disponibles dans tmrt_pred_report.")

if pred_split.empty or not {"tmrt", "tmrt_pred", "split"}.issubset(pred_split.columns):
    print("Données insuffisantes pour le nuage prédiction vs observation.")
else:
    splits = list(pred_split["split"].dropna().unique())[:4]
    ncols = min(2, len(splits))
    nrows = math.ceil(len(splits) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), squeeze=False)

    for ax, split in zip(axes.ravel(), splits):
        part = pred_split[pred_split["split"].eq(split)]
        sample = part.sample(n=min(25000, len(part)), random_state=42)
        ax.scatter(sample["tmrt"], sample["tmrt_pred"], s=3, alpha=0.25, color=BLUE)
        low = min(sample["tmrt"].min(), sample["tmrt_pred"].min())
        high = max(sample["tmrt"].max(), sample["tmrt_pred"].max())
        ax.plot([low, high], [low, high], color="black", linewidth=1)
        ax.set_title(str(split))
        ax.set_xlabel("TMRT observée (°C)")
        ax.set_ylabel("TMRT prédite (°C)")
        ax.grid(alpha=0.2)

    for ax in axes.ravel()[len(splits):]:
        ax.axis("off")

    extra_paths = [] if split_predictions_path.exists() else [FIG_EVAL / "predicted_vs_observed_available_predictions.png"]
    save_figure(fig, FIG_EVAL / "predicted_vs_observed_by_split.png", extra_paths=extra_paths)


## 3. Résidus et biais

On regarde maintenant les erreurs elles-mêmes. Une bonne moyenne globale peut masquer des biais par date, par créneau ou par parcours ; ces figures servent à repérer ces cas.


In [ ]:
residual_df = test_rows if not test_rows.empty else analysis_df

if residual_df.empty or "err" not in residual_df.columns:
    print("Aucune colonne err disponible.")
else:
    errors = residual_df["err"].dropna()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(errors, bins=80, color=BLUE, alpha=0.85)
    ax.axvline(0, color="black", linewidth=1)
    ax.axvline(errors.mean(), color=RED, linestyle="--", linewidth=1, label=f"moyenne = {errors.mean():.2f} °C")
    ax.set_xlabel("Erreur = prédiction - observation (°C)")
    ax.set_ylabel("Nombre de lignes")
    ax.set_title("Distribution des résidus")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    save_figure(fig, FIG_BIAS / "residual_distribution.png")


In [ ]:
if residual_df.empty or not {"track_id", "M_slot", "err"}.issubset(residual_df.columns):
    print("Colonnes insuffisantes pour les résidus par parcours et créneau.")
else:
    tracks = ordered_existing(residual_df["track_id"].dropna().unique(), TRACK_ORDER)
    slots = ordered_existing(residual_df["M_slot"].dropna().unique(), MSLOT_ORDER)
    fig, axes = plt.subplots(1, len(tracks), figsize=(4.6 * len(tracks), 4), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, track in zip(axes, tracks):
        part = residual_df[residual_df["track_id"].eq(track)]
        values = [part.loc[part["M_slot"].eq(slot), "err"].dropna() for slot in slots]
        boxplot_with_labels(ax, values, slots, showfliers=False)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(track)
        ax.set_xlabel("Créneau")
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("Erreur (°C)")
    fig.suptitle("Résidus par parcours et par créneau", y=1.03)
    save_figure(fig, FIG_EVAL / "residuals_by_track_mslot.png")


In [ ]:
if not bias_analysis.empty and {"group_type", "group_value", "bias_mean_pred_minus_obs"}.issubset(bias_analysis.columns):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1.5, 1, 1]})

    date_bias = bias_analysis[bias_analysis["group_type"].eq("date")].copy()
    if not date_bias.empty:
        date_bias = date_bias.sort_values("group_value")
        axes[0].barh(date_bias["group_value"], date_bias["bias_mean_pred_minus_obs"], color=BLUE)
        axes[0].axvline(0, color="black", linewidth=1)
        axes[0].set_xlabel("Biais (°C)")
        axes[0].set_title("Par date")
        axes[0].grid(axis="x", alpha=0.25)
    else:
        axes[0].axis("off")

    slot_bias = bias_analysis[bias_analysis["group_type"].eq("M_slot")].copy()
    if not slot_bias.empty:
        slot_bias["group_value"] = pd.Categorical(slot_bias["group_value"], categories=MSLOT_ORDER, ordered=True)
        slot_bias = slot_bias.sort_values("group_value")
        axes[1].bar(slot_bias["group_value"].astype(str), slot_bias["bias_mean_pred_minus_obs"], color=BLUE)
        axes[1].axhline(0, color="black", linewidth=1)
        axes[1].set_ylabel("Biais (°C)")
        axes[1].set_title("Par créneau")
        axes[1].grid(axis="y", alpha=0.25)
    else:
        axes[1].axis("off")

    track_bias = bias_analysis[bias_analysis["group_type"].eq("track_id")].copy()
    if not track_bias.empty:
        track_bias["group_value"] = pd.Categorical(track_bias["group_value"], categories=TRACK_ORDER, ordered=True)
        track_bias = track_bias.sort_values("group_value")
        axes[2].bar(track_bias["group_value"].astype(str), track_bias["bias_mean_pred_minus_obs"], color=BLUE)
        axes[2].axhline(0, color="black", linewidth=1)
        axes[2].set_ylabel("Biais (°C)")
        axes[2].set_title("Par parcours")
        axes[2].tick_params(axis="x", rotation=20)
        axes[2].grid(axis="y", alpha=0.25)
    else:
        axes[2].axis("off")

    fig.suptitle("Biais moyen de prédiction selon les groupes", y=1.03)
    save_figure(fig, FIG_BIAS / "bias_by_date_mslot_track.png")
else:
    bias_source = residual_df if not residual_df.empty else analysis_df
    if "timestamp" in bias_source.columns and "date" not in bias_source.columns:
        bias_source = bias_source.copy()
        bias_source["date"] = pd.to_datetime(bias_source["timestamp"], errors="coerce", dayfirst=True).dt.date.astype(str)

    if bias_source.empty or not {"track_id", "date", "M_slot", "err"}.issubset(bias_source.columns):
        print("Colonnes insuffisantes pour le biais par date, créneau et parcours.")
    else:
        bias_grid = (
            bias_source.dropna(subset=["track_id", "date", "M_slot", "err"])
            .groupby(["track_id", "date", "M_slot"], as_index=False)["err"]
            .mean()
        )
        tracks = ordered_existing(bias_grid["track_id"].unique(), TRACK_ORDER)
        vmax = np.nanpercentile(np.abs(bias_grid["err"]), 95)
        vmax = max(1.0, min(float(vmax), 12.0))

        fig, axes = plt.subplots(1, len(tracks), figsize=(4.8 * len(tracks), 6), sharey=True)
        axes = np.atleast_1d(axes)

        for ax, track in zip(axes, tracks):
            part = bias_grid[bias_grid["track_id"].eq(track)]
            pivot = part.pivot(index="date", columns="M_slot", values="err")
            pivot = pivot.reindex(columns=[s for s in MSLOT_ORDER if s in pivot.columns]).sort_index()
            im = ax.imshow(pivot.values, cmap="coolwarm", vmin=-vmax, vmax=vmax, aspect="auto")
            ax.set_title(track)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index)
            ax.set_xlabel("Créneau")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Biais (°C)")

        axes[0].set_ylabel("Date")
        fig.suptitle("Biais moyen par date, parcours et créneau", y=1.02)
        save_figure(fig, FIG_BIAS / "bias_by_date_mslot_track.png")


## 4. Zones où le modèle est fragile

Ces figures aident à identifier les situations les plus difficiles : valeurs de TMRT extrêmes, groupes avec forte erreur et localisation approximative des résidus.


In [ ]:
if residual_df.empty or not {"tmrt", "err"}.issubset(residual_df.columns):
    print("Colonnes tmrt et err absentes.")
else:
    tmp = residual_df.dropna(subset=["tmrt", "err"]).copy()
    tmp["classe_tmrt"] = pd.qcut(tmp["tmrt"], q=8, duplicates="drop")
    err_bins = (
        tmp.groupby("classe_tmrt", observed=True)
        .agg(mae=("err", lambda x: np.mean(np.abs(x))), rmse=("err", lambda x: np.sqrt(np.mean(x**2))))
        .reset_index()
    )
    labels = [f"{interval.left:.1f}-{interval.right:.1f}" for interval in err_bins["classe_tmrt"]]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(labels, err_bins["mae"], color=BLUE, label="MAE")
    ax.plot(labels, err_bins["rmse"], color=ORANGE, marker="o", label="RMSE")
    ax.set_xlabel("Classe de TMRT observée (°C)")
    ax.set_ylabel("Erreur (°C)")
    ax.set_title("Erreur selon le niveau de TMRT")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

    save_figure(fig, FIG_MODEL / "error_by_tmrt_range.png")


In [ ]:
if group_errors.empty or not {"track_id", "M_slot", "section_id", "rmse"}.issubset(group_errors.columns):
    print("Table des erreurs par groupe absente ou incomplète.")
else:
    top = group_errors.sort_values("rmse", ascending=False).head(20).copy()
    labels = top.apply(lambda r: f"{r['track_id']} / {r['M_slot']} / section {int(r['section_id'])}", axis=1)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(labels[::-1], top["rmse"][::-1], color=BLUE)
    ax.set_xlabel("RMSE (°C)")
    ax.set_title("Groupes avec les erreurs les plus fortes")
    ax.grid(axis="x", alpha=0.25)

    save_figure(fig, FIG_MODEL / "worst_groups_rmse.png")


In [ ]:
spatial_source = residual_df if not residual_df.empty else analysis_df
lon_col = "lon_map" if "lon_map" in spatial_source.columns else "lon_ontrack" if "lon_ontrack" in spatial_source.columns else None
lat_col = "lat_map" if "lat_map" in spatial_source.columns else "lat_ontrack" if "lat_ontrack" in spatial_source.columns else None

if spatial_source.empty or lon_col is None or lat_col is None or "err" not in spatial_source.columns:
    print("Coordonnées ou résidus absents pour la carte des erreurs.")
else:
    available = spatial_source.dropna(subset=[lon_col, lat_col, "err"])
    plot_df = available.sample(n=min(30000, len(available)), random_state=42)
    vmax = np.nanpercentile(np.abs(plot_df["err"]), 98)
    vmax = max(1.0, min(float(vmax), 12.0))

    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        plot_df[lon_col],
        plot_df[lat_col],
        c=plot_df["err"],
        s=4,
        alpha=0.65,
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Résidus spatialisés")
    plt.colorbar(sc, ax=ax, label="Erreur (°C)")

    save_figure(fig, FIG_MODEL / "spatial_residual_map.png")


## 5. Structure non indépendante des observations

Les lignes PICOPATT sont nombreuses, mais elles proviennent de passages successifs sur les mêmes parcours. Ces figures rappellent que les observations sont structurées par blocs spatiaux et temporels.


In [ ]:
if analysis_df.empty or not {"track_id", "date", "M_slot"}.issubset(analysis_df.columns):
    print("Colonnes insuffisantes pour compter les groupes de passage.")
else:
    counts = (
        analysis_df.groupby(["track_id", "date", "M_slot"])
        .size()
        .reset_index(name="n_points")
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(counts["n_points"], bins=25, color=BLUE, alpha=0.85)
    axes[0].set_xlabel("Nombre de points dans un passage")
    axes[0].set_ylabel("Nombre de passages")
    axes[0].set_title("Taille des groupes track/date/créneau")
    axes[0].grid(axis="y", alpha=0.25)

    if "track_id" in counts.columns:
        tracks = ordered_existing(counts["track_id"].unique(), TRACK_ORDER)
        data = [counts.loc[counts["track_id"].eq(track), "n_points"] for track in tracks]
        boxplot_with_labels(axes[1], data, tracks, showfliers=False)
        axes[1].set_xlabel("Parcours")
        axes[1].set_ylabel("Nombre de points")
        axes[1].set_title("Taille des passages par parcours")
        axes[1].grid(axis="y", alpha=0.25)
    else:
        axes[1].axis("off")

    save_figure(fig, FIG_NO_IID / "points_by_group.png")
    display(counts["n_points"].describe().to_frame("n_points"))


In [ ]:
def autocorr(values, lag):
    if len(values) <= lag:
        return np.nan
    a = values[:-lag]
    b = values[lag:]
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

if analysis_df.empty or not {"track_id", "date", "M_slot", "tmrt"}.issubset(analysis_df.columns):
    print("Colonnes insuffisantes pour l'autocorrélation.")
else:
    sort_cols = [c for c in ["uid", "section_id", "point_id"] if c in analysis_df.columns]
    lags = np.arange(1, 51)
    rows = []

    for _, group in analysis_df.dropna(subset=["tmrt"]).groupby(["track_id", "date", "M_slot"]):
        if sort_cols:
            group = group.sort_values(sort_cols)
        values = group["tmrt"].to_numpy(dtype=float)
        if len(values) < 20:
            continue
        for lag in lags:
            rows.append({"lag": lag, "corr": autocorr(values, lag)})

    corr_df = pd.DataFrame(rows).dropna()
    if corr_df.empty:
        print("Autocorrélation non calculable sur les groupes disponibles.")
    else:
        summary = (
            corr_df.groupby("lag")["corr"]
            .agg(mean="mean", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
            .reset_index()
        )

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(summary["lag"], summary["mean"], color=BLUE, marker="o", markersize=3, label="moyenne")
        ax.fill_between(summary["lag"], summary["q25"], summary["q75"], color=BLUE, alpha=0.18, label="Q25-Q75")
        ax.axhline(0, color="black", linewidth=1)
        ax.set_xlabel("Lag entre points successifs")
        ax.set_ylabel("Autocorrélation de la TMRT")
        ax.set_title("Autocorrélation intra-passage")
        ax.grid(alpha=0.25)
        ax.legend()

        save_figure(fig, FIG_NO_IID / "autocorrelation_tmrt_lag.png")


## Figures produites

La cellule suivante liste les fichiers enregistrés pendant l'exécution du notebook.


In [ ]:
if GENERATED_FIGURES:
    generated = pd.DataFrame({"figure": [str(rel(path)) for path in GENERATED_FIGURES]})
    display(generated)
else:
    print("Aucune figure n'a encore été générée dans cette session.")
